In [1]:
import requests
import pandas as pd
import time
import os

# ============================================================
# WORLD BANK HEALTH + SOCIOECONOMIC DATASET
# ============================================================

START_YEAR = 1960
END_YEAR = 2024

# Keep indicators with at least 30% non-missing values
MIN_COMPLETENESS = 30


# ============================================================
# INDICATORS
# ============================================================

indicators = {

    # --------------------------------------------------------
    # 1. LIFE EXPECTANCY & MORTALITY
    # --------------------------------------------------------

    "SP.DYN.LE00.IN": "Life_Expectancy",
    "SP.DYN.IMRT.IN": "Infant_Mortality",
    "SH.DYN.MORT": "Under5_Mortality",
    "SH.DYN.NMRT": "Neonatal_Mortality",
    "SH.STA.MMRT": "Maternal_Mortality",
    "SP.DYN.CDRT.IN": "Death_Rate",

    # --------------------------------------------------------
    # 2. FERTILITY & BIRTH
    # --------------------------------------------------------

    "SP.DYN.TFRT.IN": "Fertility_Rate",
    "SP.DYN.CBRT.IN": "Birth_Rate",
    "SP.ADO.TFRT": "Adolescent_Fertility",

    # --------------------------------------------------------
    # 3. POPULATION
    # --------------------------------------------------------

    "SP.POP.TOTL": "Population",
    "SP.POP.GROW": "Population_Growth",
    "SP.URB.TOTL.IN.ZS": "Urban_Population",
    "SP.RUR.TOTL.ZS": "Rural_Population",
    "SP.POP.0014.TO.ZS": "Population_Young",
    "SP.POP.1564.TO.ZS": "Population_WorkingAge",
    "SP.POP.65UP.TO.ZS": "Population_Elderly",

    # --------------------------------------------------------
    # 4. HEALTH EXPENDITURE
    # --------------------------------------------------------

    "SH.XPD.CHEX.GD.ZS": "Health_Expenditure_GDP",
    "SH.XPD.CHEX.PC.CD": "Health_Expenditure_Per_Capita",
    "SH.XPD.OOPC.CH.ZS": "OutOfPocket_Health_Expenditure",
    "SH.XPD.GHED.GD.ZS": "Government_Health_Expenditure",

    # --------------------------------------------------------
    # 5. IMMUNIZATION
    # --------------------------------------------------------

    "SH.IMM.IDPT": "DPT_Immunization",
    "SH.IMM.MEAS": "Measles_Immunization",
    "SH.IMM.POLI": "Polio_Immunization",
    "SH.IMM.HEPB": "HepB_Immunization",
    "SH.IMM.BCG": "BCG_Immunization",

    # --------------------------------------------------------
    # 6. INFECTIOUS DISEASES
    # --------------------------------------------------------

    "SH.DYN.AIDS.ZS": "HIV_Prevalence",
    "SH.TBS.INCD": "Tuberculosis_Incidence",
    "SH.MLR.INCD.P3": "Malaria_Incidence",

    # --------------------------------------------------------
    # 7. WATER & SANITATION
    # --------------------------------------------------------

    "SH.H2O.BASW.ZS": "Basic_Water_Access",
    "SH.STA.BASS.ZS": "Basic_Sanitation_Access",
    "SH.H2O.SMDW.ZS": "Safely_Managed_Water",
    "SH.STA.SMSS.ZS": "Safely_Managed_Sanitation",

    # --------------------------------------------------------
    # 8. HEALTHCARE RESOURCES
    # --------------------------------------------------------

    "SH.MED.BEDS.ZS": "Hospital_Beds",
    "SH.MED.PHYS.ZS": "Physicians",

    # --------------------------------------------------------
    # 9. NUTRITION
    # --------------------------------------------------------

    "SH.STA.STNT.ZS": "Stunting",
    "SH.STA.WAST.ZS": "Wasting",
    "SH.STA.OWGH.ZS": "Overweight",

    # --------------------------------------------------------
    # 10. ECONOMY
    # --------------------------------------------------------

    "NY.GDP.PCAP.CD": "GDP_Per_Capita",
    "NY.GDP.PCAP.KD": "GDP_Per_Capita_Constant",
    "NY.GDP.MKTP.CD": "GDP",
    "NY.GDP.MKTP.KD.ZG": "GDP_Growth",
    "NY.GNP.PCAP.CD": "GNI_Per_Capita",

    # --------------------------------------------------------
    # 11. POVERTY & INCOME
    # --------------------------------------------------------

    "SI.POV.DDAY": "Poverty_Rate",
    "SI.POV.GINI": "Gini_Index",
    "SI.SPR.PCAP.ZG": "Income_Growth",

    # --------------------------------------------------------
    # 12. EMPLOYMENT
    # --------------------------------------------------------

    "SL.UEM.TOTL.ZS": "Unemployment_Rate",
    "SL.UEM.1524.ZS": "Youth_Unemployment",
    "SL.EMP.TOTL.SP.ZS": "Employment_Rate",

    # --------------------------------------------------------
    # 13. EDUCATION
    # --------------------------------------------------------

    "SE.ADT.LITR.ZS": "Adult_Literacy",
    "SE.PRM.ENRR": "Primary_Enrollment",
    "SE.SEC.ENRR": "Secondary_Enrollment",
    "SE.TER.ENRR": "Tertiary_Enrollment",

    # --------------------------------------------------------
    # 14. ELECTRICITY & INFRASTRUCTURE
    # --------------------------------------------------------

    "EG.ELC.ACCS.ZS": "Electricity_Access",
    "EG.CFT.ACCS.ZS": "Clean_Cooking_Access",

    # --------------------------------------------------------
    # 15. ENVIRONMENT
    # --------------------------------------------------------

    "EN.ATM.PM25.MC.M3": "PM25_Air_Pollution",
    "EN.ATM.CO2E.PC": "CO2_Emissions_Per_Capita",
    "EG.USE.PCAP.KG.OE": "Energy_Use_Per_Capita",

    # --------------------------------------------------------
    # 16. TECHNOLOGY
    # --------------------------------------------------------

    "IT.NET.USER.ZS": "Internet_Users",
    "IT.CEL.SETS.P2": "Mobile_Subscriptions",

    # --------------------------------------------------------
    # 17. LIFE EXPECTANCY BY SEX
    # --------------------------------------------------------

    "SP.DYN.LE00.FE.IN": "Female_Life_Expectancy",
    "SP.DYN.LE00.MA.IN": "Male_Life_Expectancy",
}


# ============================================================
# FUNCTION TO DOWNLOAD ONE INDICATOR
# ============================================================

def download_indicator(indicator_code, indicator_name):

    print(f"Downloading: {indicator_name}")

    url = (
        f"https://api.worldbank.org/v2/"
        f"country/all/indicator/{indicator_code}"
    )

    params = {
        "format": "json",
        "date": f"{START_YEAR}:{END_YEAR}",
        "per_page": 20000
    }

    try:

        response = requests.get(
            url,
            params=params,
            timeout=60
        )

        response.raise_for_status()

        result = response.json()

        # Check whether data exists
        if len(result) < 2 or result[1] is None:

            print("   No data found")

            return None

        records = result[1]

        rows = []

        for record in records:

            country = record.get("country", {}).get("value")
            country_code = record.get("countryiso3code")
            year = record.get("date")
            value = record.get("value")

            # Ignore records without valid year
            if year is None:
                continue

            rows.append({
                "Country": country,
                "Country_Code": country_code,
                "Year": int(year),
                indicator_name: value
            })

        df = pd.DataFrame(rows)

        print(f"   Downloaded {len(df)} rows")

        return df

    except Exception as e:

        print(f"   ERROR: {e}")

        return None


# ============================================================
# DOWNLOAD ALL INDICATORS
# ============================================================

all_data = []

print("=" * 70)
print("STARTING WORLD BANK DATA DOWNLOAD")
print("=" * 70)

print(f"Years: {START_YEAR} - {END_YEAR}")
print(f"Number of indicators requested: {len(indicators)}")

for code, name in indicators.items():

    df = download_indicator(code, name)

    if df is not None:
        all_data.append(df)

    # Small delay
    time.sleep(0.5)


# ============================================================
# CHECK DATA
# ============================================================

if len(all_data) == 0:

    raise Exception(
        "No data was downloaded. "
        "Please check your internet connection."
    )


print("\n" + "=" * 70)
print("DOWNLOAD SUMMARY")
print("=" * 70)

print(
    "Number of successfully downloaded indicators:",
    len(all_data)
)

print(
    "Number of requested indicators:",
    len(indicators)
)


# ============================================================
# MERGE DATASETS
# ============================================================

print("\n" + "=" * 70)
print("MERGING DATASETS")
print("=" * 70)

merged_df = all_data[0]

for df in all_data[1:]:

    merged_df = pd.merge(
        merged_df,
        df,
        on=[
            "Country",
            "Country_Code",
            "Year"
        ],
        how="outer"
    )


# ============================================================
# SORT DATA
# ============================================================

merged_df = merged_df.sort_values(
    ["Country", "Year"]
).reset_index(drop=True)


# ============================================================
# REMOVE DUPLICATES
# ============================================================

before_duplicates = len(merged_df)

merged_df = merged_df.drop_duplicates(
    subset=[
        "Country",
        "Country_Code",
        "Year"
    ]
)

after_duplicates = len(merged_df)

print(
    f"Duplicate rows removed: "
    f"{before_duplicates - after_duplicates}"
)


# ============================================================
# CHECK INDICATOR COMPLETENESS
# ============================================================

print("\n" + "=" * 70)
print("CHECKING INDICATOR COMPLETENESS")
print("=" * 70)

data_columns = [
    column
    for column in merged_df.columns
    if column not in [
        "Country",
        "Country_Code",
        "Year"
    ]
]

completeness = (
    merged_df[data_columns]
    .notna()
    .mean()
    * 100
)

print("\nCompleteness percentage:")

print(
    completeness
    .sort_values(ascending=False)
    .round(2)
)


# ============================================================
# KEEP GOOD INDICATORS
# ============================================================

good_columns = completeness[
    completeness >= MIN_COMPLETENESS
].index.tolist()

removed_columns = completeness[
    completeness < MIN_COMPLETENESS
].index.tolist()


print("\n" + "=" * 70)
print("INDICATOR SELECTION")
print("=" * 70)

print(
    "Indicators before filtering:",
    len(data_columns)
)

print(
    "Indicators kept:",
    len(good_columns)
)

print(
    "Indicators removed:",
    len(removed_columns)
)


merged_df = merged_df[
    [
        "Country",
        "Country_Code",
        "Year"
    ] + good_columns
]


# ============================================================
# IMPORTANT:
# DO NOT REMOVE ROWS BASED ON ROW COMPLETENESS
# ============================================================

print("\n" + "=" * 70)
print("ROW COMPLETENESS")
print("=" * 70)

print(
    "All available country-year rows are being kept."
)

print(
    "Missing values will be handled during "
    "the data-cleaning stage."
)


# ============================================================
# RESET INDEX
# ============================================================

merged_df = merged_df.reset_index(drop=True)


# ============================================================
# CREATE OUTPUT DIRECTORY
# ============================================================

output_directory = "../data/raw"

os.makedirs(
    output_directory,
    exist_ok=True
)


# ============================================================
# SAVE DATASET
# ============================================================

filename = os.path.join(
    output_directory,
    "health_raw_1960_2024_extended.csv"
)

merged_df.to_csv(
    filename,
    index=False
)


# ============================================================
# RESULTS
# ============================================================

print("\n" + "=" * 70)
print("DOWNLOAD COMPLETED!")
print("=" * 70)

print("\nFile saved as:")

print(filename)

print("\nDataset shape:")

print(f"Rows    : {merged_df.shape[0]}")
print(f"Columns : {merged_df.shape[1]}")


# ============================================================
# COUNTRIES AND YEARS
# ============================================================

print("\nDataset coverage:")

print(
    "Number of countries:",
    merged_df["Country"].nunique()
)

print(
    "Minimum year:",
    merged_df["Year"].min()
)

print(
    "Maximum year:",
    merged_df["Year"].max()
)


# ============================================================
# SUCCESSFULLY KEPT INDICATORS
# ============================================================

print("\n" + "=" * 70)
print("SUCCESSFULLY KEPT INDICATORS")
print("=" * 70)

for column in good_columns:

    print(" -", column)


# ============================================================
# REMOVED INDICATORS
# ============================================================

print("\n" + "=" * 70)
print("REMOVED INDICATORS (<30% COMPLETENESS)")
print("=" * 70)

if len(removed_columns) == 0:

    print("None")

else:

    for column in removed_columns:

        print(" -", column)


# ============================================================
# FIRST 10 ROWS
# ============================================================

print("\n" + "=" * 70)
print("FIRST 10 ROWS")
print("=" * 70)

print(
    merged_df.head(10)
)


# ============================================================
# MISSING VALUES
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUES")
print("=" * 70)

missing_values = (
    merged_df
    .isnull()
    .sum()
    .sort_values(ascending=False)
)

print(missing_values)


# ============================================================
# DATASET INFORMATION
# ============================================================

print("\n" + "=" * 70)
print("DATASET INFORMATION")
print("=" * 70)

merged_df.info()


# ============================================================
# FINAL MESSAGE
# ============================================================

print("\n" + "=" * 70)
print("DATA COLLECTION FINISHED SUCCESSFULLY")
print("=" * 70)

print(
    "\nYour raw dataset is ready for the next stage."
)

print(
    "\nNext step:"
)

print(
    "02_data_validation.ipynb"
)

STARTING WORLD BANK DATA DOWNLOAD
Years: 1960 - 2024
Number of indicators requested: 61
Downloading: Life_Expectancy
   Downloaded 17225 rows
Downloading: Infant_Mortality
   Downloaded 17225 rows
Downloading: Under5_Mortality
   Downloaded 17225 rows
Downloading: Neonatal_Mortality
   Downloaded 17225 rows
Downloading: Maternal_Mortality
   Downloaded 17225 rows
Downloading: Death_Rate
   Downloaded 17225 rows
Downloading: Fertility_Rate
   Downloaded 17225 rows
Downloading: Birth_Rate
   Downloaded 17225 rows
Downloading: Adolescent_Fertility
   Downloaded 17225 rows
Downloading: Population
   Downloaded 17225 rows
Downloading: Population_Growth
   Downloaded 17225 rows
Downloading: Urban_Population
   Downloaded 17225 rows
Downloading: Rural_Population
   Downloaded 17225 rows
Downloading: Population_Young
   Downloaded 17225 rows
Downloading: Population_WorkingAge
   Downloaded 17225 rows
Downloading: Population_Elderly
   Downloaded 17225 rows
Downloading: Health_Expenditure_GDP
 

In [3]:
filename = "../data/raw/health_raw_1960_2024_extended.csv"

merged_df.to_csv(
    filename,
    index=False
)

In [4]:
file_path = "../data/raw/health_raw_1960_2024_extended.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (17225, 51)
